<a href="https://colab.research.google.com/github/GiovanniMerici/PhyloProf/blob/main/PhyloProf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Import
!pip install biopython
!pip install ete3
!apt-get install ncbi-blast+ -y


from ete3 import NCBITaxa
from Bio import Phylo
from Bio import Entrez
from Bio import SeqIO
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import time
import glob
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
#@title Function

organisms = [
    "Homo sapiens",
    "Mus musculus",
    "Danio Rerio"

]


os.makedirs("genomes", exist_ok=True)


def download_genome(organism, max_results=1):
    print(f"\nSearching for: {organism}")
    search_term = f'"{organism}"[Organism] AND "latest refseq"[filter]'

    handle = Entrez.esearch(db="assembly", term=search_term, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()

    id_list = record["IdList"]
    if not id_list:
        print(f"No genome found for {organism}")
        return

    for uid in id_list:
        summary = Entrez.read(Entrez.esummary(db="assembly", id=uid, report="full"))
        ftp_path = summary['DocumentSummarySet']['DocumentSummary'][0]['FtpPath_RefSeq']
        if not ftp_path:
            print(f"No FTP path for {organism}")
            continue

        # Build the .fna.gz file path
        base = os.path.basename(ftp_path)
        fasta_url = f"{ftp_path}/{base}_genomic.fna.gz"
        fasta_url = f"{fasta_url.split('_genomic')[0]}_protein.faa.gz"


        output_path = f"genomes/{organism.replace(' ', '_')}.faa.gz"
        print(f"Downloading: {fasta_url}")
        !curl -s "{fasta_url}" -o "{output_path}"


        time.sleep(1)

def make_blast_databases(folder="genomes"):
    for filename in os.listdir(folder):
        if filename.endswith(".faa"):
            filepath = os.path.join(folder, filename)
            print(f"Creating BLAST DB for {filename}")
            dbpath = os.path.splitext(filepath)[0]
            !makeblastdb -in "{filepath}" -dbtype prot -out "{dbpath}"


def length_genome(input_file):
  len_seq = []
  name_seq = []
  for rec in SeqIO.parse(input_file,'fasta'):
    name_seq.append(rec.id)
    len_seq.append(len(rec.seq))

  len_dict = dict(zip(name_seq,len_seq))
  return len_dict

def profiler(genomes, len_dict):
  df_f = pd.DataFrame()
  gen_order= []
  no_res = 0
  no_res_name = []
  gene_list = list(len_dict.keys())
  for i, genome in enumerate(genomes):
    try:
        genome_name = os.path.basename(genome).split('.faa')[0].replace("_", " ")
        df = pd.read_table(f"result_{os.path.basename(genome).split('.faa')[0]}.txt", header=None)
        #identity --> 80% (filter 1)
        df = df[df[2] >0]
        df['length'] = df[0].map(len_dict)
        df['coverage'] = df[3]/df['length']
        #coverage --> 0.8 (filter 2)
        df = df[df['coverage'] > 0.8]
        df = df.groupby(0)[1].count().reset_index().T
        new_header = df.iloc[0]
        df = df[1:]
        df.columns = new_header
        df_f = pd.concat([df_f,df])
        gen_order.append(os.path.basename(genome).split('.faa')[0])
    except:
        print(f"No results for {os.path.basename(genome).split('.faa')[0]}")
        no_res = no_res + 1
        no_res_name.append(os.path.basename(genome).split('.faa')[0].replace("_", " "))




  gen_order = [x.replace("_"," ") for x in gen_order]
  df_f.index = gen_order
  df_f.fillna(0, inplace=True)

  for i  in range(no_res):
    df_f.loc[no_res_name[i]] = [0] * df_f.shape[1]

  df_f = df_f.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
  df_f.to_csv("profile.csv")
  return df_f

def get_taxid(name):
    print(f"Searching TaxId for: {name}")
    handle = Entrez.esearch(db="taxonomy", term=name)
    record = Entrez.read(handle)
    if record['IdList']:
        return int(record['IdList'][0])
    time.sleep(1)
    return None

def plot_tree_and_heatmap(taxids, faa_files, len_dict, output_file="tree_with_heatmap.png"):

    # Get and rename NCBI taxonomy tree
    ncbi = NCBITaxa()
    tree = ncbi.get_topology(list(taxids.values()), intermediate_nodes=True)
    for node in tree.traverse():
        if node.is_leaf():
            for org, taxid in taxids.items():
                if node.name == str(taxid):
                    node.name = org.replace(" ", "_")

    # Save tree
    tree.write(outfile="tree.nwk", format=1)

    # Load tree
    phylo_tree = Phylo.read("tree.nwk", "newick")

    # Generate gene profile
    df_profile = profiler(faa_files, len_dict)

    # Align matrix with tree leaf order
    leaf_order = [term.name.replace("_", " ") for term in phylo_tree.get_terminals()]
    leaf_order = [l for l in leaf_order if l in df_profile.index]
    df_profile = df_profile.loc[leaf_order]

    # Create figure
    fig = plt.figure(figsize=(16, 10))
    ax1 = fig.add_axes([0.05, 0.1, 0.3, 0.8])
    ax2 = fig.add_axes([0.4, 0.1, 0.55, 0.8])

    # Plot tree
    Phylo.draw(phylo_tree, axes=ax1, do_show=False, show_confidence=False)
    x0, x1 = ax1.get_xlim()
    ax1.set_xlim(x0, x1 + 25)
    ax1.set_title("Phylogenetic Tree", fontsize=12)

    # Plot heatmap
    sns.heatmap(
        df_profile,
        cmap="viridis",
        ax=ax2,
        cbar_kws={'label': 'Presence (1) / Absence (0)'},
        annot=False,
        cbar=False,
        linecolor='white',
        linewidths=0.5
    )

    # Label formatting
    ax2.set_title("Phylogenetic Profile")
    ax2.set_xlabel("Genes")
    ax2.set_ylabel("Organisms")
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=90, ha='right')

    # Save and show
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

def extract_organism(org_inp):
  user_org = []
  with open(org_inp,'r') as f:
    for line in f:
      user_org.append(line.strip())
  return user_org

def download_uniprot_fasta(uniprot_ids_file, output_file="input_from_uniprot.fasta"):

    with open(uniprot_ids_file, "r") as f:
        ids = [line.strip() for line in f if line.strip()]

    with open(output_file, "w") as out:
        for uid in ids:
            url = f"https://www.uniprot.org/uniprot/{uid}.fasta"
            r = requests.get(url)
            if r.ok and r.text.startswith(">"):
                out.write(r.text)
            else:
                print(f"[WARNING] Could not download FASTA for {uid}")

    return output_file


def get_input_fasta():

    mode = input("Do you want to upload a FASTA file or a list of UniProt IDs? (type 'fasta' or 'uniprot'): ").strip().lower()

    if mode == "fasta":
        fasta_file = input("Enter the FASTA file name: ")
        while not os.path.isfile(fasta_file):
            print("File not found.")
            fasta_file = input("Enter the FASTA file name: ")
        return fasta_file

    elif mode == "uniprot":
        uid_file = input("Enter the file with UniProt IDs (one per line): ")
        while not os.path.isfile(uid_file):
            print("File not found.")
            uid_file = input("Enter the file with UniProt IDs (one per line): ")
        return download_uniprot_fasta(uid_file)

    else:
        print("Invalid option. Please type 'fasta' or 'uniprot'.")
        return get_input_fasta()

In [ ]:
#@title Main Code

Entrez.email = input("Enter your email to download data from Entrez:")
input_file = get_input_fasta()


org_inp =  input("Enter the organism file name (enter no for default organism): ")
if org_inp == "no":
    organisms = organisms
else:
    while not os.path.isfile(org_inp):
        print("File not found.")
        org_inp = input("Enter the organism file name: ")
    organisms = extract_organism(org_inp)

for org in organisms:
    download_genome(org)
    time.sleep(1)

!gunzip genomes/*.gz

make_blast_databases()

faa_files = glob.glob('genomes/*.faa')

for faa in faa_files:
    db_path = os.path.splitext(faa)[0]
    result_file = f"result_{os.path.basename(db_path)}.txt"
    cmd = f"blastp -query {input_file} -db '{db_path}' -out '{result_file}' -outfmt 6 -max_target_seqs 1 -max_hsps 1 -evalue 1e-5"
    print(f"[RUNNING] {cmd}")
    os.system(cmd)

len_dict = length_genome(input_file)

df_profile = profiler(faa_files, len_dict)

taxids = {org: get_taxid(org) for org in organisms}

plot_tree_and_heatmap(taxids, faa_files, len_dict)